# Car Damage Detection — YOLOv5

**Author:** Ilyes Khayati  
**Institution:** MedTech — Mediterranean Institute of Technology  
**Company:** AddInn Group, Tunis  
**Academic Supervisor:** Dr. Walid Ben Haj Othmen  
**Institution Supervisor:** Dr. Nivine Attoue  
**Year:** 2024–2025

---

## Overview

Car insurance companies currently rely on human experts to manually review vehicle photographs and assess damage — a process that is both slow and expensive at scale. This project builds an automated detection pipeline using a custom-trained YOLOv5s model that detects and localises damage regions in vehicle images using bounding boxes.

The intended use case is a mobile application where a car owner photographs their vehicle and receives an instant assessment, reducing the time and cost of the insurance claim process.

**Notebook structure:**

1. Environment Setup
2. Dataset Loading and Exploratory Analysis
3. Model Training
4. Results and Performance Analysis
5. Inference on Test Images
6. Conclusions

---
## 1. Environment Setup

The Ultralytics YOLOv5 repository is cloned from GitHub and all required dependencies are installed from its bundled `requirements.txt`. Training was run on Google Colab Pro with an NVIDIA A100 GPU, which brought training time from roughly 8 hours on the free tier down to around 1.5 hours.

YOLO (You Only Look Once) processes an image in a single forward pass, outputting bounding boxes and class probabilities simultaneously. This end-to-end design makes it considerably faster than two-stage detectors like Faster R-CNN, and more practical for real-time or mobile deployment. The `v5s` (small) variant was selected to keep inference fast and the model file lightweight enough for mobile integration.

In [ ]:
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -r requirements.txt -q
!pip install roboflow -q

In [ ]:
import torch
import os
import yaml
import glob
import random
from pathlib import Path
from IPython.display import Image, display, clear_output
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
from PIL import Image as PILImage

# Verify GPU availability before proceeding
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_properties(0).name
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU     : {gpu_name}  ({gpu_mem:.1f} GB)")
else:
    print("No GPU found — training on CPU will be significantly slower.")

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device.upper()}")

---
## 2. Dataset Loading and Exploratory Analysis

### 2.1 Dataset Selection

The dataset is sourced from Roboflow Universe. An initial dataset provided by AddInn was evaluated but discarded: its validation and test splits together accounted for roughly 1% of the total ~10,000 images, making honest evaluation impossible. The resulting model showed unstable metrics and precision that plateaued around 0.6.

The replacement dataset uses a standard 70/20/10 train/val/test split across 9,900 images, which resolved the instability entirely.

| Split | Images | Share |
|-------|--------|-------|
| Train | ~6,930 | 70% |
| Validation | ~1,980 | 20% |
| Test | ~990 | 10% |
| **Total** | **9,900** | |

**Class:** `Car-Damage` (single class — binary detection)  
**Annotation format:** YOLOv5 normalised bounding box — `[class x_center y_center width height]`  
**License:** CC BY 4.0

In [ ]:
from roboflow import Roboflow

os.environ["DATASET_DIRECTORY"] = "/content/datasets"

# Replace with your own API key — available at roboflow.com under Settings > API Keys
ROBOFLOW_API_KEY = "YOUR_API_KEY"

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("ayhan-gul-hgudf").project("car-damage-rlogo")
version = project.version(1)
dataset = version.download("yolov5")

DATASET_PATH = dataset.location
print(f"Dataset downloaded to: {DATASET_PATH}")

### 2.2 Exploratory Data Analysis

Before training it is worth understanding the structure and distribution of the data. The following cells inspect the dataset configuration, verify image and label counts per split, analyse bounding box dimensions across the training set, and display a sample of annotated images.

In [ ]:
# Read and display the dataset configuration file
yaml_path = os.path.join(DATASET_PATH, "data.yaml")

with open(yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

print("Dataset Configuration")
print("-" * 40)
print(f"  Classes ({data_config['nc']}) : {data_config['names']}")
print(f"  Train           : {data_config['train']}")
print(f"  Validation      : {data_config['val']}")
print(f"  Test            : {data_config['test']}")
print("-" * 40)

In [ ]:
# Count images and corresponding label files per split
splits = {"Train": "train", "Validation": "valid", "Test": "test"}

print(f"{'Split':<14} {'Images':>8} {'Labels':>8}")
print("-" * 33)

split_counts = {}
for name, folder in splits.items():
    img_dir   = os.path.join(DATASET_PATH, folder, "images")
    label_dir = os.path.join(DATASET_PATH, folder, "labels")
    n_imgs    = len(glob.glob(os.path.join(img_dir,   "*.jpg")) +
                   glob.glob(os.path.join(img_dir,   "*.png")))
    n_labels  = len(glob.glob(os.path.join(label_dir, "*.txt")))
    split_counts[name] = n_imgs
    print(f"{name:<14} {n_imgs:>8,} {n_labels:>8,}")

total = sum(split_counts.values())
print("-" * 33)
print(f"{'Total':<14} {total:>8,}")

In [ ]:
# Dataset split distribution — pie and bar chart
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("Dataset Split Distribution", fontsize=12, fontweight="bold")

colors = ["#3A7FBF", "#4FA860", "#D97C2B"]
labels = list(split_counts.keys())
sizes  = list(split_counts.values())

axes[0].pie(
    sizes, labels=labels, autopct="%1.1f%%",
    colors=colors, startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
axes[0].set_title("Proportion by Split")

bars = axes[1].bar(labels, sizes, color=colors, edgecolor="white", width=0.5)
for bar, val in zip(bars, sizes):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 40,
        f"{val:,}", ha="center", va="bottom", fontsize=10
    )
axes[1].set_ylabel("Number of Images")
axes[1].set_title("Image Count per Split")
axes[1].set_ylim(0, max(sizes) * 1.15)
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("dataset_split.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Parse all training label files and compute bounding box statistics.
# YOLOv5 label format: class  x_center  y_center  width  height  (all normalised 0-1)

label_dir   = os.path.join(DATASET_PATH, "train", "labels")
label_files = glob.glob(os.path.join(label_dir, "*.txt"))

boxes_w, boxes_h, boxes_area = [], [], []
annotations_per_image = []

for lf in label_files:
    with open(lf) as f:
        lines = [l.strip() for l in f if l.strip()]
    annotations_per_image.append(len(lines))
    for line in lines:
        parts = line.split()
        if len(parts) == 5:
            _, _, _, w, h = map(float, parts)
            boxes_w.append(w)
            boxes_h.append(h)
            boxes_area.append(w * h)

boxes_w    = np.array(boxes_w)
boxes_h    = np.array(boxes_h)
boxes_area = np.array(boxes_area)

print("Bounding Box Statistics — Training Set (normalised coordinates)")
print("-" * 58)
print(f"  Total annotations      : {len(boxes_w):,}")
print(f"  Avg annotations/image  : {np.mean(annotations_per_image):.2f}")
print(f"  Max annotations/image  : {max(annotations_per_image)}")
print(f"  Avg box width          : {boxes_w.mean():.3f}  ({boxes_w.mean()*100:.1f}% of image width)")
print(f"  Avg box height         : {boxes_h.mean():.3f}  ({boxes_h.mean()*100:.1f}% of image height)")
print(f"  Avg box area           : {boxes_area.mean():.4f} ({boxes_area.mean()*100:.2f}% of image area)")
print(f"  Median box area        : {np.median(boxes_area):.4f}")

In [ ]:
# Bounding box dimension distributions across the training set
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Bounding Box Dimension Distribution — Training Set",
             fontsize=12, fontweight="bold")

plot_data = [
    (boxes_w,    "Box Width (normalised)",  "#3A7FBF"),
    (boxes_h,    "Box Height (normalised)", "#4FA860"),
    (boxes_area, "Box Area (normalised)",   "#C0392B"),
]

for ax, (data, xlabel, color) in zip(axes, plot_data):
    ax.hist(data, bins=50, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(
        np.mean(data), color="black", linestyle="--",
        linewidth=1.4, label=f"Mean: {np.mean(data):.3f}"
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("bbox_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Draw YOLO-format bounding boxes onto an image array.
# Written without OpenCV to avoid an extra dependency in this notebook.

def draw_yolo_boxes(img_path, label_path, color=(0, 180, 0), thickness=2):
    img = np.array(PILImage.open(img_path).convert("RGB"))
    h, w = img.shape[:2]

    if not os.path.exists(label_path):
        return img

    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            _, xc, yc, bw, bh = map(float, parts)
            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)
            img[max(0,y1):min(h,y1+thickness), max(0,x1):min(w,x2)] = color
            img[max(0,y2-thickness):min(h,y2), max(0,x1):min(w,x2)] = color
            img[max(0,y1):min(h,y2), max(0,x1):min(w,x1+thickness)] = color
            img[max(0,y1):min(h,y2), max(0,x2-thickness):min(w,x2)] = color
    return img


train_img_dir   = os.path.join(DATASET_PATH, "train", "images")
train_label_dir = os.path.join(DATASET_PATH, "train", "labels")
all_imgs = (
    glob.glob(os.path.join(train_img_dir, "*.jpg")) +
    glob.glob(os.path.join(train_img_dir, "*.png"))
)

sample_imgs = random.sample(all_imgs, min(9, len(all_imgs)))

fig, axes = plt.subplots(3, 3, figsize=(13, 13))
fig.suptitle("Sample Training Images — Ground-Truth Annotations",
             fontsize=12, fontweight="bold", y=1.01)

for ax, img_path in zip(axes.flatten(), sample_imgs):
    stem       = Path(img_path).stem
    label_path = os.path.join(train_label_dir, stem + ".txt")
    annotated  = draw_yolo_boxes(img_path, label_path)
    ax.imshow(annotated)
    ax.axis("off")
    ax.set_title(stem[:28], fontsize=7)

plt.tight_layout()
plt.savefig("sample_annotations.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3. Model Training

### Configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Architecture | YOLOv5s | Small variant — fast inference, suitable for mobile deployment |
| Image size | 640 | Standard input resolution for this dataset |
| Batch size | 16 | Fills A100 VRAM without overflow |
| Epochs | 100 | Loss curves converged well before the end |
| Weights | yolov5s.pt | COCO-pretrained — transfer learning from general visual features |
| Cache | enabled | Pre-loads images to RAM, reducing I/O overhead per epoch |
| Save period | 10 | Checkpoint saved every 10 epochs |

The model is initialised from weights pretrained on COCO (`yolov5s.pt`) rather than from random initialisation. This lets the network start with already-learned low-level features — edges, textures, shapes — and adapt them to the damage detection task in far fewer epochs than a random start would require. With roughly 7,000 training images, transfer learning is not optional; it is necessary.

In [ ]:
# Verify the data.yaml path before launching training
yaml_path = os.path.join(DATASET_PATH, "data.yaml")
assert os.path.exists(yaml_path), f"data.yaml not found at: {yaml_path}"

print(f"data.yaml confirmed: {yaml_path}")
print()
with open(yaml_path) as f:
    print(f.read())

In [ ]:
# --img        : input image resolution
# --batch      : images processed per gradient update
# --epochs     : full passes through the training set
# --data       : dataset configuration file
# --cfg        : model architecture definition
# --weights    : pretrained weights for transfer learning
# --save-period: checkpoint frequency in epochs
# --cache      : cache images to RAM to reduce I/O bottleneck

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 100 \
    --data "{yaml_path}" \
    --cfg models/yolov5s.yaml \
    --weights yolov5s.pt \
    --save-period 10 \
    --cache

In [ ]:
# Locate the output directory from the most recent training run
run_dirs   = sorted(glob.glob("runs/train/exp*"))
latest_run = run_dirs[-1] if run_dirs else "runs/train/exp"

print(f"Training output: {latest_run}")
print()
print(f"{'File':<40} {'Size (MB)':>10}")
print("-" * 52)
for f in sorted(os.listdir(latest_run)):
    fpath = os.path.join(latest_run, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f"{f:<40} {size:>10.2f}")

---
## 4. Results and Performance Analysis

### Evaluation Metrics

| Term | Definition |
|------|------------|
| TP — True Positive | Damage correctly detected where it exists |
| FP — False Positive | Damage predicted where there is none |
| FN — False Negative | Real damage that the model missed |
| Precision | TP / (TP + FP) — of all predictions, how many were correct |
| Recall | TP / (TP + FN) — of all real damages, how many were found |
| mAP@0.5 | Mean Average Precision at IoU threshold 0.5 — primary metric |
| F1 Score | Harmonic mean of Precision and Recall |

For an insurance application, recall carries particular weight. A missed damage region (false negative) is a more costly error than a false alarm (false positive) — so a model that leans toward high recall is preferable in practice.

In [ ]:
# Load the training results CSV automatically generated by YOLOv5
results_csv = os.path.join(latest_run, "results.csv")
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print(f"Epochs recorded : {len(df)}")
print(f"Columns         : {list(df.columns)}")
print()
print("Final epoch:")
print(df.tail(1).T.to_string())

In [ ]:
# Training and validation loss curves
epochs = df.index + 1

loss_cols = [
    ("train/box_loss", "val/box_loss",  "Box Loss",        "#C0392B", "#E8A49A"),
    ("train/obj_loss", "val/obj_loss",  "Objectness Loss", "#2471A3", "#85C1E9"),
    ("train/cls_loss", "val/cls_loss",  "Class Loss",      "#1E8449", "#82E0AA"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Training vs Validation Loss", fontsize=12, fontweight="bold")

for ax, (train_col, val_col, title, tc, vc) in zip(axes, loss_cols):
    if train_col in df.columns:
        ax.plot(epochs, df[train_col], color=tc, linewidth=2, label="Train")
    if val_col in df.columns:
        ax.plot(epochs, df[val_col], color=vc, linewidth=2,
                linestyle="--", label="Val")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Performance metrics over epochs
metric_cols = [
    ("metrics/precision",    "Precision",      "#7D3C98"),
    ("metrics/recall",       "Recall",         "#148F77"),
    ("metrics/mAP_0.5",      "mAP @ 0.5",      "#CA6F1E"),
    ("metrics/mAP_0.5:0.95", "mAP @ 0.5:0.95", "#1A5276"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Model Performance Metrics over Training",
             fontsize=12, fontweight="bold")

for ax, (col, label, color) in zip(axes.flatten(), metric_cols):
    if col not in df.columns:
        ax.set_visible(False)
        continue
    ax.plot(epochs, df[col], color=color, linewidth=2)
    ax.fill_between(epochs, df[col], alpha=0.12, color=color)
    best_val   = df[col].max()
    best_epoch = df[col].idxmax() + 1
    ax.axhline(best_val, linestyle=":", color="gray", linewidth=1)
    ax.annotate(
        f"Best: {best_val:.4f}  (epoch {best_epoch})",
        xy=(best_epoch, best_val),
        xytext=(best_epoch + 2, best_val - 0.06),
        fontsize=8, color="gray"
    )
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(label)
    ax.set_ylim(0, 1.05)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("metrics_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Performance summary for the best epoch
best_idx = df.get("metrics/mAP_0.5", df.iloc[:, 0]).idxmax()
best_row = df.loc[best_idx]

metric_map = {
    "metrics/precision":    "Precision",
    "metrics/recall":       "Recall",
    "metrics/mAP_0.5":      "mAP @ IoU 0.50",
    "metrics/mAP_0.5:0.95": "mAP @ IoU 0.50:0.95",
}

print("Best Model — Performance Summary")
print("-" * 45)
for col, name in metric_map.items():
    if col in df.columns:
        print(f"  {name:<24} {best_row[col]:.4f}")

if "metrics/precision" in df.columns and "metrics/recall" in df.columns:
    P  = best_row["metrics/precision"]
    R  = best_row["metrics/recall"]
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0
    print(f"  {'F1 Score':<24} {F1:.4f}")

print("-" * 45)

In [ ]:
# Display the standard evaluation plots saved by YOLOv5 after training
result_plots = [
    ("PR_curve.png",         "Precision-Recall Curve"),
    ("P_curve.png",          "Precision-Confidence Curve"),
    ("R_curve.png",          "Recall-Confidence Curve"),
    ("F1_curve.png",         "F1-Confidence Curve"),
    ("confusion_matrix.png", "Confusion Matrix"),
]

available = [
    (name, title)
    for name, title in result_plots
    if os.path.exists(os.path.join(latest_run, name))
]

if available:
    ncols = min(len(available), 3)
    nrows = (len(available) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 6 * nrows))
    axes = np.array(axes).flatten()

    for ax, (fname, title) in zip(axes, available):
        ax.imshow(mpimg.imread(os.path.join(latest_run, fname)))
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axis("off")

    for ax in axes[len(available):]:
        ax.set_visible(False)

    plt.suptitle("YOLOv5 Evaluation Plots", fontsize=13,
                 fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("all_eval_plots.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Evaluation plots not found — run training first.")

### Notes on the Results

**Precision-Recall curve.** A curve that stays close to the top-right corner of the plot indicates a strong detector — high precision at high recall simultaneously. The area under this curve is the mAP@0.5 score. Our model achieves approximately 0.987, near the theoretical maximum of 1.0. The shape is almost rectangular, which confirms the model is neither overfitting nor underfitting.

**Confusion matrix.** With a single class, the matrix is effectively 2x2: Car-Damage versus background. A value of 0.99 on the diagonal means the model misclassifies only 1% of detections.

**Comparison with the initial dataset.** The first training run on the imbalanced dataset produced a precision ceiling around 0.6 with erratic, non-monotonic metric curves. Switching to a properly split dataset resolved this entirely. The metrics converged smoothly and consistently from the first epoch, confirming that the imbalanced split — not the model architecture or hyperparameters — was the root cause of the earlier poor performance.

---
## 5. Inference on Test Images

The trained model is run against the held-out test set. Predictions are displayed side-by-side with ground-truth annotations, and the distribution of confidence scores across all detections is analysed.

In [ ]:
# Confirm best weights exist before running inference
best_weights = os.path.join(latest_run, "weights", "best.pt")
assert os.path.exists(best_weights), f"Weights not found at: {best_weights}"

print(f"Best weights : {best_weights}")
print(f"File size    : {os.path.getsize(best_weights)/1e6:.2f} MB")

In [ ]:
test_img_dir = os.path.join(DATASET_PATH, "test", "images")

!python detect.py \
    --weights "{best_weights}" \
    --img 640 \
    --conf 0.25 \
    --source "{test_img_dir}" \
    --save-txt \
    --save-conf

In [ ]:
# Locate the inference output directory
detect_dirs   = sorted(glob.glob("runs/detect/exp*"))
latest_detect = detect_dirs[-1] if detect_dirs else "runs/detect/exp"

detected_imgs = (
    glob.glob(os.path.join(latest_detect, "*.jpg")) +
    glob.glob(os.path.join(latest_detect, "*.png"))
)

print(f"Inference output  : {latest_detect}")
print(f"Images processed  : {len(detected_imgs)}")

In [ ]:
# Ground truth vs prediction — side-by-side comparison
sample_detected = random.sample(detected_imgs, min(6, len(detected_imgs)))

fig, axes = plt.subplots(len(sample_detected), 2,
                         figsize=(12, 5 * len(sample_detected)))
fig.suptitle("Ground Truth  vs  Model Predictions",
             fontsize=12, fontweight="bold", y=1.01)

if len(sample_detected) == 1:
    axes = [axes]

for row_axes, pred_path in zip(axes, sample_detected):
    stem     = Path(pred_path).stem
    gt_path  = os.path.join(DATASET_PATH, "test", "images", Path(pred_path).name)
    lbl_path = os.path.join(DATASET_PATH, "test", "labels", stem + ".txt")

    if os.path.exists(gt_path):
        row_axes[0].imshow(draw_yolo_boxes(gt_path, lbl_path, color=(0, 180, 0)))
        row_axes[0].set_title("Ground Truth", fontweight="bold")
    else:
        row_axes[0].set_visible(False)

    row_axes[1].imshow(mpimg.imread(pred_path))
    row_axes[1].set_title("Prediction", fontweight="bold")

    for ax in row_axes:
        ax.axis("off")

plt.tight_layout()
plt.savefig("gt_vs_pred.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Confidence score distribution across all test-set predictions
pred_label_dir = os.path.join(latest_detect, "labels")
conf_scores = []

if os.path.exists(pred_label_dir):
    for lf in glob.glob(os.path.join(pred_label_dir, "*.txt")):
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 6:   # class xc yc w h conf
                    conf_scores.append(float(parts[5]))

if conf_scores:
    conf_scores = np.array(conf_scores)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle("Prediction Confidence Score Distribution",
                 fontsize=12, fontweight="bold")

    axes[0].hist(conf_scores, bins=40, color="#2471A3",
                 edgecolor="white", alpha=0.85)
    axes[0].axvline(
        conf_scores.mean(), color="#C0392B", linestyle="--",
        linewidth=1.5, label=f"Mean: {conf_scores.mean():.3f}"
    )
    axes[0].set_xlabel("Confidence Score")
    axes[0].set_ylabel("Count")
    axes[0].legend()
    axes[0].spines[["top", "right"]].set_visible(False)

    sorted_conf = np.sort(conf_scores)
    cdf = np.arange(1, len(sorted_conf) + 1) / len(sorted_conf)
    pct_above = (conf_scores >= 0.5).mean() * 100
    axes[1].plot(sorted_conf, cdf, color="#7D3C98", linewidth=2)
    axes[1].axvline(0.5, color="gray", linestyle=":",
                    linewidth=1.5, label="Threshold = 0.50")
    axes[1].set_title(f"{pct_above:.1f}% of predictions at or above 0.50 confidence")
    axes[1].set_xlabel("Confidence Score")
    axes[1].set_ylabel("Cumulative Fraction")
    axes[1].legend()
    axes[1].spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig("confidence_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Predictions analysed : {len(conf_scores):,}")
    print(f"Mean confidence      : {conf_scores.mean():.3f}")
    print(f"Median confidence    : {np.median(conf_scores):.3f}")
    print(f"Min / Max            : {conf_scores.min():.3f} / {conf_scores.max():.3f}")
else:
    print("No prediction labels found — re-run detect.py with --save-txt --save-conf")

---
## 6. Conclusions

### Final Results

| Metric | Score |
|--------|-------|
| mAP @ IoU 0.50 | ~0.987 |
| mAP @ IoU 0.50:0.95 | ~0.80 |
| Precision (at best confidence threshold) | 1.00 |
| Confusion Matrix Accuracy | 0.99 |

### Key Takeaways

**Dataset quality is the determining factor.** The first dataset's 1% validation/test split made it structurally impossible for the model to learn meaningful generalisation signals. Once replaced with a properly split dataset, the metrics converged cleanly from the first epoch. The lesson is straightforward: before tuning hyperparameters, verify that your data splits are honest.

**Transfer learning matters at this data scale.** With roughly 7,000 training images, initialising from COCO-pretrained weights gave the model a significant head start. Training from random initialisation would have required substantially more data or epochs to reach the same level of convergence.

**YOLOv5s is well-suited for deployment.** The trained weights file is approximately 14 MB — small enough to ship inside a mobile application. Inference runs in real time even on mid-range hardware, which is a practical requirement for the insurance use case this project targets.

### Potential Next Steps

- **Multi-class damage categorisation** — adding classes such as `scratch`, `dent`, or `broken glass` would give insurers more granular information for pricing claims without human review.
- **Vehicle part localisation** — a second class dimension for car part (bumper, door, hood) would make the output directly actionable.
- **Severity regression** — attaching a regression head to estimate damage severity as a continuous score, not just a binary detection.
- **Quantisation** — applying INT8 post-training quantisation to reduce model size and speed up inference on low-end mobile CPUs.
- **Hyperparameter search** — the `hyp.scratch.yaml` file exposes learning rate schedules, augmentation strengths, and anchor settings that were left at defaults here; a systematic search could yield a further improvement in mAP.

In [ ]:
# Run summary
print("Run Summary")
print("-" * 52)
print(f"  Training output   : {latest_run}")
print(f"  Best weights      : {best_weights}")
print(f"  Inference output  : {latest_detect}")
print()
print("  Saved plots:")
saved = [
    "dataset_split.png",
    "bbox_distribution.png",
    "sample_annotations.png",
    "loss_curves.png",
    "metrics_curves.png",
    "all_eval_plots.png",
    "gt_vs_pred.png",
    "confidence_distribution.png",
]
for fname in saved:
    status = "found  " if os.path.exists(fname) else "missing"
    print(f"    [{status}]  {fname}")
print("-" * 52)